# PairRecon pair queries

This notebook reads one Pair from `pairs.xlsx` and runs ad-hoc SQL against its Left and Right sources through PairRecon's own connections: the Left through `source_from_config(...).query`, a Databricks Right through the SQL warehouse (`StatementSql`), a CSV Right through DuckDB. `{left}` and `{right}` in a query are replaced by the side's relation (table, `query` subquery, or CSV reader). Results come back inline, so keep large queries aggregated or add a `LIMIT`.

In [ ]:
from pathlib import Path

import pandas as pd

from pair_recon.app.config import env_value, load_environment
from pair_recon.app.runner import StatementSql, pair_config, workspace
from pair_recon.core.metadata.workbook import read_pair_workbook
from pair_recon_left.backends.sources import source_from_config
from pair_recon_left.common.sql.dates import quote_ident
from pair_recon_right.engine.backends.targets import SparkTarget

load_environment()
PROFILE = env_value("DBX_PROFILE", default="corp")
WAREHOUSE_ID = env_value("DBX_WAREHOUSE_ID", default="")
WORKBOOK = Path(env_value("DBX_PAIR_WORKBOOK", default="setup/pairs.xlsx")).expanduser()
PAIR_NAME = ""
FROM_DATE = None
TO_DATE = None

In [ ]:
rows = {row["pair_name"]: row for row in read_pair_workbook(WORKBOOK)}
if PAIR_NAME not in rows:
    raise ValueError(f"PAIR_NAME must be one of: {', '.join(sorted(rows))}")
effective = pair_config(rows[PAIR_NAME])
partition = str(effective.get("partition") or "day")
left_config = {**effective["left"], "partition": partition}
right_config = {**effective["right"], "partition": partition}
left = source_from_config(left_config)
right_is_csv = str(right_config.get("source") or "").casefold() == "csv"
right = source_from_config(right_config) if right_is_csv else SparkTarget(None, right_config)
right_dialect = right.dialect if right_is_csv else "spark"
sql = None if right_is_csv else StatementSql(workspace(PROFILE), WAREHOUSE_ID)
print(f"partition={partition}")
print(f"left  {left_config.get('source')}: {left.relation(from_date=FROM_DATE, to_date=TO_DATE)}")
print(f"right {right_config.get('source')}: {right.relation(from_date=FROM_DATE, to_date=TO_DATE)}")

In [ ]:
# >>> source WHERE from the pair config, optionally ANDed with an extra predicate <<< #
def where(config, extra=None):
    clauses = [item for item in (config.get("where"), extra) if item]
    return " AND ".join(f"({item})" for item in clauses) or "1=1"


# >>> fill {left}/{right} with each side's relation for the current FROM_DATE/TO_DATE <<< #
def render(statement):
    return statement.format(
        left=left.relation(from_date=FROM_DATE, to_date=TO_DATE),
        right=right.relation(from_date=FROM_DATE, to_date=TO_DATE),
    )


# >>> run SQL on the Left engine and return a DataFrame <<< #
def run_left(statement):
    columns, values = left.query(render(statement), operation="notebook")
    return pd.DataFrame([tuple(item) for item in values], columns=columns)


# >>> run SQL on the Right engine (SQL warehouse or DuckDB for CSV) and return a DataFrame <<< #
def run_right(statement):
    if right_is_csv:
        columns, values = right.query(render(statement), operation="notebook")
        return pd.DataFrame([tuple(item) for item in values], columns=columns)
    return pd.DataFrame(sql.execute(render(statement)))


# >>> MIN/MAX/COUNT of a side's date_col (or any column) under the pair's WHERE <<< #
def date_range(side, column=None):
    config, dialect, runner = (
        (left_config, left.dialect, run_left) if side == "left" else (right_config, right_dialect, run_right)
    )
    name = quote_ident(column or config["date_col"], dialect)
    return runner(
        f"SELECT MIN({name}) AS min_value, MAX({name}) AS max_value, COUNT(*) AS row_count "
        f"FROM {{{side}}} WHERE {where(config)}"
    )

In [ ]:
display(date_range("left"))
display(date_range("right"))

Ad-hoc queries: write SQL in each engine's dialect and use `{left}` / `{right}` for the relation. `where(left_config)` / `where(right_config)` return the pair's configured filter.

In [ ]:
display(run_left(f"SELECT COUNT(*) AS n FROM {{left}} WHERE {where(left_config)}"))
display(run_right(f"SELECT COUNT(*) AS n FROM {{right}} WHERE {where(right_config)}"))